<a href="https://colab.research.google.com/github/jesicaroman17-cpu/Intro_ciencia-datos-JR/blob/main/Tarea1/homework01_Jesica_Roman_Martinez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 6 · Tarea 01 — 8 ejercicios autocalificables de pandas

### Series, DataFrames, filtrado, groupby y más

**Instrucciones**

1. Implementa cada función reemplazando `raise NotImplementedError`.
2. **No cambies el nombre ni los parámetros** de las funciones.
3. Ejecuta las celdas de **tests visibles** y **tests adicionales**. Si una función
   está bien, verás ✅; si no, saltará un `AssertionError`.
4. Tu objetivo: que **todas** las celdas de tests pasen sin error.

> 🗂️ Cada ejercicio carga el dataset desde `../datasets/transacciones.csv`.

> ⚠️ Recién abierta, esta tarea **no** corre de principio a fin: las celdas de tests
> fallarán hasta que implementes cada función. ¡Eso es lo que vas a arreglar!

In [ ]:
# Identifícate (esto ayuda si entregas el notebook).
NOMBRE = "Jesica Roman Matinez"   # ✏️ escribe tu nombre completo
print("Tarea lista para resolver. Ejecuta cada bloque de tests tras implementar.")

## Ejercicio 1 · Porcentaje de participación por ciudad en ventas totales

Implementa `participacion_ciudades(df)` que reciba el DataFrame de
transacciones y devuelva un **DataFrame** con columnas:
- `'ciudad'`: nombre de la ciudad
- `'total'`: suma de montos de esa ciudad
- `'pct'`: porcentaje del total global, redondeado a 2 decimales

Ordenado de mayor a menor `'pct'`.

**Ejemplo:** si Bogota tiene el 25% de las ventas, `pct` = 25.0

In [1]:
def participacion_ciudades(df):
  df_nuevo = df.groupby('ciudad', as_index=False).agg(total=('monto', 'sum'))
  df_nuevo['pct'] = round(df_nuevo['total'] / df_nuevo['total'].sum() * 100, 2)
  return df_nuevo.sort_values('pct', ascending=False)

In [2]:
# === Tests visibles · Ejercicio 1 ===
import os, pandas as pd
_df = pd.read_csv('/content/transacciones.csv')
_res = participacion_ciudades(_df)
assert 'ciudad' in _res.columns
assert 'total' in _res.columns
assert 'pct' in _res.columns
assert len(_res) == 5
assert abs(_res['pct'].sum() - 100.0) < 0.1
print("✅ Ejercicio 1: tests visibles superados.")

✅ Ejercicio 1: tests visibles superados.


In [3]:
# === Tests adicionales (ocultos) · Ejercicio 1 ===
assert _res.iloc[0]['pct'] >= _res.iloc[-1]['pct']
assert (_res['pct'] > 0).all()
assert abs(_res['total'].sum() - _df['monto'].sum()) < 1
print("✅ Ejercicio 1: tests adicionales superados.")

✅ Ejercicio 1: tests adicionales superados.


## Ejercicio 2 · Categoría con mayor variabilidad (desviación estándar) de precios

Implementa `categoria_mas_variable(df)` que devuelva el **nombre**
(str) de la categoría con mayor desviación estándar de montos.

**Ejemplo:** devuelve algo como `'tecnologia'` (el nombre exacto
depende del dataset).

In [4]:
def categoria_mas_variable(df):
  desviaciones = df.groupby('categoria')['monto'].std()
  return desviaciones.idxmax()

In [6]:
# === Tests visibles · Ejercicio 2 ===
import os, pandas as pd
_df = pd.read_csv('/content/transacciones.csv')
_cat = categoria_mas_variable(_df)
assert isinstance(_cat, str)
assert _cat in _df['categoria'].unique()
print("✅ Ejercicio 2: tests visibles superados.")

✅ Ejercicio 2: tests visibles superados.


In [7]:
# === Tests adicionales (ocultos) · Ejercicio 2 ===
_stds = _df.groupby('categoria')['monto'].std()
assert _cat == _stds.idxmax()
print("✅ Ejercicio 2: tests adicionales superados.")

✅ Ejercicio 2: tests adicionales superados.


## Ejercicio 3 · Función resumen_ciudad(df, ciudad) que devuelva métricas clave

Implementa `resumen_ciudad(df, ciudad)` que devuelva una **Series**
con las siguientes métricas para la ciudad indicada:
- `'n_transacciones'`: número de transacciones
- `'total'`: suma de montos
- `'promedio'`: promedio de monto (float)
- `'maximo'`: monto máximo
- `'minimo'`: monto mínimo

**Ejemplo:** `resumen_ciudad(df, 'Bogota')['n_transacciones']` → número entero.

In [8]:
import pandas as pd

def resumen_ciudad(df, ciudad):
  montos_ciudad = df[df['ciudad'] == ciudad]['monto']

  return pd.Series({
      'n_transacciones': montos_ciudad.count(),
      'total': montos_ciudad.sum(),
      'promedio': montos_ciudad.mean(),
      'maximo': montos_ciudad.max(),
      'minimo': montos_ciudad.min()
      })

In [12]:
# === Tests visibles · Ejercicio 3 ===
import os, pandas as pd
_df = pd.read_csv('/content/transacciones.csv')
_s = resumen_ciudad(_df, 'Bogota')
assert hasattr(_s, 'index')
assert 'n_transacciones' in _s.index
assert 'total' in _s.index
assert 'promedio' in _s.index
print("✅ Ejercicio 3: tests visibles superados.")

✅ Ejercicio 3: tests visibles superados.


In [13]:
# === Tests adicionales (ocultos) · Ejercicio 3 ===
assert _s['n_transacciones'] == len(_df[_df['ciudad'] == 'Bogota'])
assert _s['total'] == _df[_df['ciudad'] == 'Bogota']['monto'].sum()
assert _s['minimo'] <= _s['promedio'] <= _s['maximo']
print("✅ Ejercicio 3: tests adicionales superados.")

✅ Ejercicio 3: tests adicionales superados.


## Ejercicio 4 · Detectar y eliminar transacciones duplicadas por id

Implementa `eliminar_duplicados(df)` que devuelva una **copia** del
DataFrame sin filas con `id` duplicado. Si hay duplicados, se conserva
la **primera** aparición.

La función debe devolver el DataFrame limpio (sin modificar el original).

In [14]:
def eliminar_duplicados(df):
  df_limpio = df.drop_duplicates(subset='id', keep='first')
  return df_limpio

In [15]:
# === Tests visibles · Ejercicio 4 ===
import os, pandas as pd
_df = pd.read_csv('/content/transacciones.csv')
_df_dup = pd.concat([_df, _df.head(3)], ignore_index=True)
_df_limpio = eliminar_duplicados(_df_dup)
assert len(_df_limpio) == len(_df)
assert _df_limpio['id'].nunique() == len(_df_limpio)
print("✅ Ejercicio 4: tests visibles superados.")

✅ Ejercicio 4: tests visibles superados.


In [16]:
# === Tests adicionales (ocultos) · Ejercicio 4 ===
assert len(_df_dup) == len(_df) + 3
assert list(_df_limpio['id'][:3]) == list(_df['id'][:3])
print("✅ Ejercicio 4: tests adicionales superados.")

✅ Ejercicio 4: tests adicionales superados.


## Ejercicio 5 · Ventas acumuladas (cumsum) ordenadas por fecha

Implementa `ventas_acumuladas(df)` que devuelva una **copia** del
DataFrame, ordenada por fecha (ascendente), con una columna adicional
`'acumulado'` que contenga la suma acumulada de monto.

El índice del resultado debe ser 0, 1, 2, ... (reset_index).

In [17]:
def ventas_acumuladas(df):
  df_acumulado = df.sort_values('fecha').reset_index(drop=True)
  df_acumulado['acumulado'] = df_acumulado['monto'].cumsum()
  return df_acumulado

In [21]:
# === Tests visibles · Ejercicio 5 ===
import os, pandas as pd
_df = pd.read_csv('/content/transacciones.csv', parse_dates=['fecha'])
_dfac = ventas_acumuladas(_df)
assert 'acumulado' in _dfac.columns
assert len(_dfac) == len(_df)
assert _dfac['acumulado'].iloc[-1] == _df['monto'].sum()
print("✅ Ejercicio 5: tests visibles superados.")

✅ Ejercicio 5: tests visibles superados.


In [22]:
# === Tests adicionales (ocultos) · Ejercicio 5 ===
assert (_dfac['fecha'].diff().dropna() >= pd.Timedelta(0)).all()
assert (_dfac['acumulado'].diff().dropna() >= 0).all()
print("✅ Ejercicio 5: tests adicionales superados.")

✅ Ejercicio 5: tests adicionales superados.


## Ejercicio 6 · Comparar medianas por método de pago con groupby + agg

Implementa `comparar_medianas(df)` que devuelva un **DataFrame** con
columnas `'metodo_pago'`, `'mediana'` y `'promedio'` (redondeados a
2 decimales), ordenado de mayor a menor `'mediana'`.

In [24]:
def comparar_medianas(df):
  resultado = df.groupby('metodo_pago', as_index=False).agg(mediana=('monto', 'median'), promedio=('monto', 'mean'))
  resultado['mediana'] = resultado['mediana'].round(2)
  resultado['promedio'] = resultado['promedio'].round(2)
  return resultado.sort_values('mediana', ascending=False).reset_index(drop=True)

In [25]:
# === Tests visibles · Ejercicio 6 ===
import os, pandas as pd
_df = pd.read_csv('/content/transacciones.csv')
_res = comparar_medianas(_df)
assert 'metodo_pago' in _res.columns
assert 'mediana' in _res.columns
assert 'promedio' in _res.columns
assert len(_res) == 3
print("✅ Ejercicio 6: tests visibles superados.")

✅ Ejercicio 6: tests visibles superados.


In [26]:
# === Tests adicionales (ocultos) · Ejercicio 6 ===
assert _res.iloc[0]['mediana'] >= _res.iloc[-1]['mediana']
assert (_res['mediana'] > 0).all()
assert (_res['promedio'] > 0).all()
print("✅ Ejercicio 6: tests adicionales superados.")

✅ Ejercicio 6: tests adicionales superados.


## Ejercicio 7 · Filtrar y exportar transacciones atípicas (z-score > 2)

Implementa `exportar_atipicos(df, ruta_salida)` que:
1. Calcule el z-score de la columna `monto`:
   `z = (monto - monto.mean()) / monto.std()`
2. Filtre las filas donde `abs(z) > 2`.
3. Exporte esas filas a `ruta_salida` como CSV (sin índice).
4. Devuelva el número de filas exportadas.

**Ejemplo:** devuelve un entero con el número de outliers encontrados.

In [27]:
def exportar_atipicos(df, ruta_salida):
  z_score = (df['monto'] - df['monto'].mean()) / df['monto'].std()
  atipicos = df[abs(z_score) > 2]
  atipicos.to_csv(ruta_salida, index=False)
  return len(atipicos)

In [29]:
# === Tests visibles · Ejercicio 7 ===
import os, pandas as pd, tempfile
_df = pd.read_csv('/content/transacciones.csv')
with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as _f: _ruta_tmp = _f.name
_n = exportar_atipicos(_df, _ruta_tmp)
assert isinstance(_n, int)
assert _n >= 0
print("✅ Ejercicio 7: tests visibles superados.")

✅ Ejercicio 7: tests visibles superados.


In [30]:
# === Tests adicionales (ocultos) · Ejercicio 7 ===
_df_out = pd.read_csv(_ruta_tmp)
assert len(_df_out) == _n
_z = (_df['monto'] - _df['monto'].mean()) / _df['monto'].std()
_expected = int((_z.abs() > 2).sum())
assert _n == _expected
os.unlink(_ruta_tmp)
print("✅ Ejercicio 7: tests adicionales superados.")

✅ Ejercicio 7: tests adicionales superados.


## Ejercicio 8 · Crear columna 'trimestre' a partir de fecha y agrupar

Implementa `ventas_por_trimestre(df)` que:
1. Agregue una columna `'trimestre'` con el trimestre del año (1, 2, 3 o 4),
   calculado como `((mes - 1) // 3) + 1` donde `mes = fecha.dt.month`.
2. Devuelva un **DataFrame** con columnas `'trimestre'` y `'total'`
   (suma de montos por trimestre), ordenado por trimestre.

**Ejemplo:** `trimestre=1` es enero-marzo, `trimestre=2` es abril-junio, etc.

In [31]:
def ventas_por_trimestre(df):
  df_trimestre = df.copy()
  mes = df_trimestre['fecha'].dt.month
  df_trimestre['trimestre'] = ((mes - 1) // 3) + 1
  resultado = df_trimestre.groupby('trimestre', as_index=False).agg(total=('monto', 'sum'))
  return resultado.sort_values('trimestre').reset_index(drop=True)

In [33]:
# === Tests visibles · Ejercicio 8 ===
import os, pandas as pd
_df = pd.read_csv('/content/transacciones.csv', parse_dates=['fecha'])
_res = ventas_por_trimestre(_df)
assert 'trimestre' in _res.columns
assert 'total' in _res.columns
assert len(_res) >= 1
print("✅ Ejercicio 8: tests visibles superados.")

✅ Ejercicio 8: tests visibles superados.


In [34]:
# === Tests adicionales (ocultos) · Ejercicio 8 ===
assert _res['trimestre'].between(1, 4).all()
assert abs(_res['total'].sum() - _df['monto'].sum()) < 1
print("✅ Ejercicio 8: tests adicionales superados.")

✅ Ejercicio 8: tests adicionales superados.


---
## Entrega

Cuando **todas** las celdas de tests muestren ✅, has completado la tarea.

Repaso de conceptos ejercitados:

| Ejercicio | Concepto pandas |
|---|---|
| 1 Participación por ciudad | `groupby` + porcentaje |
| 2 Categoría más variable | `groupby` + `std` + `idxmax` |
| 3 Resumen de ciudad | filtrado + `pd.Series` manual |
| 4 Eliminar duplicados | `drop_duplicates` |
| 5 Ventas acumuladas | `sort_values` + `cumsum` |
| 6 Comparar medianas | `groupby` + `agg` con lambda |
| 7 Exportar atípicos | z-score + filtrado + `to_csv` |
| 8 Agrupar por trimestre | `dt.month` + `groupby` |

> 💭 **Reflexión:** el z-score del ejercicio 7 es una técnica estadística simple
> pero poderosa. En machine learning se usa para normalizar features. ¿Puedes
> pensarlo en términos del patrón "acumulador + transformación vectorizada"?